In [ ]:
import pandas as pd
import os
import pydicom
from glob import glob
import re
from functools import partial
from PIL import Image
from tqdm import tqdm
tqdm.pandas()

In [ ]:
def get_one_nion(x, outdir):
    slides = glob(f"/nfs/turbo/umms-tocho/root_srh_db/UM/{x['type_institution_number']}*/*/mosaic*/*1.dcm")
    slides = [f for f in slides if re.search(f"{x['type_institution_number']}[a-zA-Z]*/", f)]

    new_dir = ["{}/NIO_UM_{}.{}.tiff".format(outdir, *re.findall(".*/NIO_UM_([a-zA-Z0-9]+)/([a-zA-Z0-9]+)/.*", t)[0]) for t in slides]
    for src, tgt in zip(slides, new_dir):
        Image.fromarray(pydicom.dcmread(src).pixel_array).save(tgt)       

In [ ]:
hl_data = pd.read_csv("HollonLabDatabase.csv")

In [ ]:
umich_nio = hl_data[(hl_data["type_institution_number"].str.contains("NIO")) & (hl_data["Institution_shortcode"]=="UM")]

In [ ]:
umich_nio_glioma = umich_nio.loc[umich_nio["tumor type "]=='glioma']

In [ ]:
gbm = umich_nio_glioma[umich_nio_glioma['IDH1 status '].isin({"1. wildtype"})]
idhmut = umich_nio_glioma[umich_nio_glioma['IDH1 status '].isin({"2. R132H","3. R132C", "imputed_idhmut"})]
oligo = idhmut[idhmut['1p/19q co-deletion ']=="2. negative"]
astro = idhmut[idhmut['1p/19q co-deletion ']=="1. positive"]

In [ ]:
gbms = gbm.sample(n=40, random_state=1000).sort_index()
gbms.to_csv("gbm_scbench.csv", index=False)
astros = astro.sample(n=40, random_state=1000).sort_index()
astros.to_csv("astro_scbench.csv", index=False)
oligos = oligo.sample(n=40, random_state=1000).sort_index()
oligos.to_csv("oligo_scbench.csv", index=False)


In [ ]:
os.makedirs("scbench_gbm", exist_ok=True)
gbms.progress_apply(partial(get_one_nion, outdir="scbench_gbm"), axis=1)
os.makedirs("scbench_astro", exist_ok=True)
astros.progress_apply(partial(get_one_nion, outdir="scbench_astro"), axis=1)
os.makedirs("scbench_oligo", exist_ok=True)
oligos.progress_apply(partial(get_one_nion, outdir="scbench_oligo"), axis=1)

In [ ]:
umich_nio["tumor type "].value_counts()

In [ ]:
umich_nio_pit = umich_nio.loc[umich_nio["tumor type "]=="pituitary adenoma"]
pits = umich_nio_pit.sample(n=40, random_state=1000).sort_index()
pits.to_csv("pit_scbench.csv", index=False)
os.makedirs("scbench_pit", exist_ok=True)
pits.progress_apply(partial(get_one_nion, outdir="scbench_pit"), axis=1)
print("OK")

In [ ]:
umich_nio_mening = umich_nio.loc[umich_nio["tumor type "]=="meningioma"]
menings = umich_nio_mening.sample(n=40, random_state=1000).sort_index()
menings.to_csv("mening_scbench.csv", index=False)
os.makedirs("scbench_mening", exist_ok=True)
menings.progress_apply(partial(get_one_nion, outdir="scbench_mening"), axis=1)
print("OK")

In [ ]:
umich_nio_schwan = umich_nio.loc[umich_nio["tumor type "]=="schwannoma"]
schwans = umich_nio_schwan.sample(n=40, random_state=1000).sort_index()
schwans.to_csv("schwan_scbench.csv", index=False)
os.makedirs("scbench_schwan", exist_ok=True)
schwans.progress_apply(partial(get_one_nion, outdir="scbench_schwan"), axis=1)
print("OK")

In [ ]:
embryonal = umich_nio.loc[umich_nio["tumor type "]=="embryonal"].sort_index()
embryonal.to_csv("embryonal_scbench.csv", index=False)
os.makedirs("scbench_embryonal", exist_ok=True)
embryonal.progress_apply(partial(get_one_nion, outdir="scbench_embryonal"), axis=1)
print("OK")

In [ ]:
umich_nio_normal = umich_nio.loc[umich_nio["tumor type "]=="normal"]
normals = umich_nio_normal.sample(n=40, random_state=1000).sort_index()
normals.to_csv("normal_scbench.csv", index=False)
os.makedirs("scbench_normal", exist_ok=True)
normals.progress_apply(partial(get_one_nion, outdir="scbench_normal"), axis=1)
print("OK")

In [ ]:
met = umich_nio[umich_nio["tumor type "] == "metastatic carcinoma"]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 100)
melanoma_mask = met["permanent diagnosis "].str.lower().str.contains("melanoma").fillna(False)
squamous_mask = met["permanent diagnosis "].str.lower().str.contains("squamous").fillna(False)
adeno_mask = met["permanent diagnosis "].str.lower().str.contains("adenocarcinoma").fillna(False)
sarcoma_mask = met["permanent diagnosis "].str.lower().str.contains("sarcoma").fillna(False)

In [ ]:
umich_nio_melanoma = met[melanoma_mask & ~squamous_mask & ~adeno_mask & ~sarcoma_mask]
melanomas = umich_nio_melanoma.sample(n=20, random_state=1000).sort_index()
melanomas.to_csv("melanoma_scbench.csv", index=False)

squamous = met[~melanoma_mask & squamous_mask & ~adeno_mask & ~sarcoma_mask].sort_index()
squamous.to_csv("squamous_scbench.csv", index=False)

umich_nio_adeno = met[~melanoma_mask & ~squamous_mask & adeno_mask & ~sarcoma_mask]
adenos = umich_nio_adeno.sample(n=20, random_state=1000).sort_index()
adenos.to_csv("adeno_scbench.csv", index=False)

sarcomas = met[~melanoma_mask & ~squamous_mask & ~adeno_mask & sarcoma_mask].sort_index()
sarcomas.to_csv("sarcomas_scbench.csv", index=False)

os.makedirs("scbench_melanoma", exist_ok=True)
melanomas.progress_apply(partial(get_one_nion, outdir="scbench_melanoma"), axis=1)
os.makedirs("scbench_squamous", exist_ok=True)
squamous.progress_apply(partial(get_one_nion, outdir="scbench_squamous"), axis=1)
os.makedirs("scbench_adeno", exist_ok=True)
adenos.progress_apply(partial(get_one_nion, outdir="scbench_adeno"), axis=1)
os.makedirs("scbench_sarcoma", exist_ok=True)
sarcomas.progress_apply(partial(get_one_nion, outdir="scbench_sarcoma"), axis=1)
print("OK")

In [ ]:
get_one_nion({"type_institution_number":"NIO_UM_827"}, outdir=".")